# 使用 MyScale 作为 OpenAI 嵌入的向量数据库

本 notebook 分步介绍如何使用 MyScale 作为 OpenAI 嵌入的向量数据库。流程包括：

1. 使用由 OpenAI API 生成的预计算嵌入。
2. 将这些嵌入存储到 MyScale 的云实例中。
3. 使用 OpenAI API 将原始文本查询转换为嵌入。
4. 利用 MyScale 在所创建的集合中执行最近邻搜索。

### 什么是 MyScale

[MyScale](https://myscale.com) 是一个基于 Clickhouse 构建的数据库，它将向量搜索与 SQL 分析相结合，提供高性能、精简且完全托管的体验。它旨在支持对结构化数据和向量数据进行联合查询与分析，并为所有数据处理提供全面的 SQL 支持。


### 部署选项

- 使用 [MyScale Console](https://console.myscale.com)，在两分钟内于你的集群上部署并通过 SQL 执行向量搜索。


## 前提条件

要跟随本指南操作，你需要准备以下内容：

1. 按照[快速入门指南](https://docs.myscale.com/en/quickstart/)部署的 MyScale 集群。
2. 用于与 MyScale 交互的 'clickhouse-connect' 库。
3. 用于将查询向量化的 [OpenAI API 密钥](https://beta.openai.com/account/api-keys)。

### 安装依赖

本 notebook 需要 `openai`、`clickhouse-connect` 以及其他一些依赖。使用以下命令安装：

In [ ]:
! pip install openai clickhouse-connect wget pandas

### 准备 OpenAI API 密钥

要使用 OpenAI API，你需要设置 API 密钥。如果还没有，可以从 [OpenAI](https://platform.openai.com/account/api-keys) 获取。

In [ ]:
import openai

# get API key from on OpenAI website
openai.api_key = "OPENAI_API_KEY"

# check we have authenticated
openai.Engine.list()

## 连接到 MyScale

按照[连接详情](https://docs.myscale.com/en/cluster-management/)一节，从 MyScale 控制台获取集群主机、用户名和密码信息，并按如下所示使用它们创建与集群的连接：

In [1]:
import clickhouse_connect

# initialize client
client = clickhouse_connect.get_client(host='YOUR_CLUSTER_HOST', port=8443, username='YOUR_USERNAME', password='YOUR_CLUSTER_PASSWORD')

## 加载数据

我们需要加载 OpenAI 提供的 Wikipedia 文章预计算向量嵌入数据集。使用 `wget` 包下载该数据集。

In [ ]:
import wget

embeddings_url = "https://cdn.openai.com/API/examples/data/vector_database_wikipedia_articles_embedded.zip"

# The file is ~700 MB so this will take some time
wget.download(embeddings_url)

下载完成后，使用 `zipfile` 包解压文件：

In [ ]:
import zipfile

with zipfile.ZipFile("vector_database_wikipedia_articles_embedded.zip", "r") as zip_ref:
    zip_ref.extractall("../data")

现在，我们可以将 `vector_database_wikipedia_articles_embedded.csv` 中的数据加载到 Pandas DataFrame 中：

In [ ]:
import pandas as pd

from ast import literal_eval

# read data from csv
article_df = pd.read_csv('../data/vector_database_wikipedia_articles_embedded.csv')
article_df = article_df[['id', 'url', 'title', 'text', 'content_vector']]

# read vectors from strings back into a list
article_df["content_vector"] = article_df.content_vector.apply(literal_eval)
article_df.head()

## 索引数据

我们将在 MyScale 中创建一个名为 `articles` 的 SQL 表来存储嵌入数据。该表将包含一个使用余弦距离度量的向量索引，以及对嵌入长度的约束。使用以下代码创建 articles 表并插入数据：

In [ ]:
# create articles table with vector index
embedding_len=len(article_df['content_vector'][0]) # 1536

client.command(f"""
CREATE TABLE IF NOT EXISTS default.articles
(
    id UInt64,
    url String,
    title String,
    text String,
    content_vector Array(Float32),
    CONSTRAINT cons_vector_len CHECK length(content_vector) = {embedding_len},
    VECTOR INDEX article_content_index content_vector TYPE HNSWFLAT('metric_type=Cosine')
)
ENGINE = MergeTree ORDER BY id
""")

# insert data into the table in batches
from tqdm.auto import tqdm

batch_size = 100
total_records = len(article_df)

# upload data in batches
data = article_df.to_records(index=False).tolist()
column_names = article_df.columns.tolist() 

for i in tqdm(range(0, total_records, batch_size)):
    i_end = min(i + batch_size, total_records)
    client.insert("default.articles", data[i:i_end], column_names=column_names)

由于向量索引会在后台自动构建，在进行搜索之前，我们需要检查其构建状态。

In [2]:
# check count of inserted data
print(f"articles count: {client.command('SELECT count(*) FROM default.articles')}")

# check the status of the vector index, make sure vector index is ready with 'Built' status
get_index_status="SELECT status FROM system.vector_indices WHERE name='article_content_index'"
print(f"index build status: {client.command(get_index_status)}")

articles count: 25000
index build status: Built


## 搜索数据

在 MyScale 中完成索引后，我们就可以执行向量搜索来查找相似内容。首先，使用 OpenAI API 为查询生成嵌入。然后，使用 MyScale 执行向量搜索。

In [4]:
import openai

query = "Famous battles in Scottish history"

# creates embedding vector from user query
embed = openai.Embedding.create(
    input=query,
    model="text-embedding-3-small",
)["data"][0]["embedding"]

# query the database to find the top K similar content to the given query
top_k = 10
results = client.query(f"""
SELECT id, url, title, distance(content_vector, {embed}) as dist
FROM default.articles
ORDER BY dist
LIMIT {top_k}
""")

# display results
for i, r in enumerate(results.named_results()):
    print(i+1, r['title'])

1 Battle of Bannockburn
2 Wars of Scottish Independence
3 1651
4 First War of Scottish Independence
5 Robert I of Scotland
6 841
7 1716
8 1314
9 1263
10 William Wallace
